# 01 - Explore the sources
Quick look at the EIA and World Bank APIs before relying on them: which endpoints exist, how Egypt is filtered, which units come back.
Not part of the pipeline; the real code is in `ingestion/`. It reads the key from `eia_key.txt`.
**Never paste the API key into a cell or commit outputs that show it.**

In [ ]:
import requests, pandas as pd
from pathlib import Path
KEY = Path("../eia_key.txt").read_text().strip()   # same key file the script uses
EIA = "https://api.eia.gov/v2"
WB = "https://api.worldbank.org/v2"
def eia(path, **params):
    r = requests.get(f"{EIA}/{path}", params={"api_key": KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()


## EIA: what does the `international` route offer?

In [ ]:
meta = eia("international/")
print(meta["response"].get("description"))
print([f["id"] for f in meta["response"]["facets"]], meta["response"]["frequency"])

## EIA: facet values (look for Egypt, and for the energy products / activities you need)

In [ ]:
for facet in ["countryRegionId", "productId", "activityId", "unit"]:
    rows = eia(f"international/facet/{facet}")["response"]["facets"]
    print(facet, len(rows))
pd.DataFrame(eia("international/facet/productId")["response"]["facets"])

## EIA: a small Egypt pull. Check the field names, the value type and the units

In [ ]:
body = eia("international/data/", **{"frequency": "annual", "data[0]": "value",
        "facets[countryRegionId][]": "EGY", "length": 20})
print("total rows:", body["response"]["total"])
df = pd.DataFrame(body["response"]["data"])
df.head(20)

In [ ]:
# which product / activity / unit combinations exist for Egypt? Copy what you need into config.yaml (eia.facets)
full = pd.DataFrame(eia("international/data/", **{"frequency": "annual", "data[0]": "value",
        "facets[countryRegionId][]": "EGY", "length": 5000})["response"]["data"])
full.groupby(["productId", "productName", "activityId", "activityName", "unit"]).size()

## World Bank: the three indicators

In [ ]:
for ind in ["EG.FEC.RNEW.ZS", "EG.ELC.ACCS.ZS", "EG.USE.PCAP.KG.OE"]:
    meta, rows = requests.get(f"{WB}/country/EGY/indicator/{ind}", params={"format": "json", "per_page": 100}, timeout=30).json()
    d = pd.DataFrame(rows)
    print(ind, "| rows:", meta["total"], "| last updated:", meta["lastupdated"], "| non-null:", d["value"].notna().sum())
d.head()